In [0]:
from pyspark.sql.types import *
from datetime import datetime
import uuid

# ---- Where tables and files live ----
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.rag")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.rag.raw")
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA rag")

# ---- Parameters (this is what makes backfills possible) ----
dbutils.widgets.text("load_type", "full")          # full or incremental
dbutils.widgets.text("start_date", "2024-10-01")
dbutils.widgets.text("end_date", "2026-10-10")
dbutils.widgets.text("base_path", "/Volumes/workspace/rag/raw")
dbutils.widgets.text("batch_id", "")
dbutils.widgets.text(
    "repos",
    "langchain-ai/langchain,run-llama/llama_index,chroma-core/chroma,qdrant/qdrant,deepset-ai/haystack"
)

LOAD_TYPE  = dbutils.widgets.get("load_type")
START_DATE = dbutils.widgets.get("start_date")
END_DATE   = dbutils.widgets.get("end_date")
BASE_PATH  = dbutils.widgets.get("base_path")
REPOS      = [r.strip() for r in dbutils.widgets.get("repos").split(",")]
BATCH_ID   = dbutils.widgets.get("batch_id") or (
    datetime.now().strftime("%Y%m%d%H%M%S") + "_" + uuid.uuid4().hex[:6]
)

# ---- Audit log table ----
LOG_SCHEMA = StructType([
    StructField("run_id", StringType()),
    StructField("batch_id", StringType()),
    StructField("layer", StringType()),
    StructField("load_type", StringType()),
    StructField("parameter_or_file", StringType()),
    StructField("start_time", TimestampType()),
    StructField("end_time", TimestampType()),
    StructField("status", StringType()),
    StructField("rows_inserted", LongType()),
    StructField("rows_updated", LongType()),
    StructField("error_message", StringType()),
])

spark.sql("""
CREATE TABLE IF NOT EXISTS pipeline_execution_logs (
  run_id STRING, batch_id STRING, layer STRING, load_type STRING,
  parameter_or_file STRING, start_time TIMESTAMP, end_time TIMESTAMP,
  status STRING, rows_inserted BIGINT, rows_updated BIGINT, error_message STRING
) USING DELTA
""")

def log_run(layer, param, start, status, inserted=0, updated=0, error=None):
    row = [(uuid.uuid4().hex, BATCH_ID, layer, LOAD_TYPE, param, start,
            datetime.now(), status, int(inserted), int(updated), error)]
    spark.createDataFrame(row, LOG_SCHEMA).write.mode("append") \
         .saveAsTable("pipeline_execution_logs")

print("Config ready. Batch:", BATCH_ID, "| Path:", BASE_PATH)